In [ ]:
# Imports and reproducibility
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
from tensorflow.keras.datasets import fashion_mnist

(x_train_k, y_train_k), (x_test_k, y_test_k) = fashion_mnist.load_data()

print("Keras load shapes:", x_train_k.shape, y_train_k.shape, x_test_k.shape, y_test_k.shape)

In [ ]:
notebook_dir = Path.cwd()
archive_path = notebook_dir / "fashion_mnist_local.npz"

np.savez_compressed(
    archive_path,
    x_train=x_train_k,
    y_train=y_train_k,
    x_test=x_test_k,
    y_test=y_test_k,
)

print("Saved local bundle (overwrite each run is fine):", archive_path)

In [ ]:
if not archive_path.is_file():
    raise FileNotFoundError(
        "Run the previous cells once so Keras can fill fashion_mnist_local.npz, "
        "or copy that archive from another machine next to this notebook."
    )

bundle = np.load(archive_path)
x_train = bundle["x_train"]
y_train = bundle["y_train"]
x_test = bundle["x_test"]
y_test = bundle["y_test"]

class_names = [
    "T-shirt",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

print("Local load shapes:", x_train.shape, y_train.shape)

In [ ]:
# Scale pixels to 0–1 and add channel dimension for Conv2D
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

num_classes = len(class_names)
input_shape = x_train.shape[1:]

In [ ]:
# Visual sanity check
figure, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()
for index in range(10):
    axes[index].imshow(x_train[index].squeeze(), cmap="gray")
    label_index = int(y_train[index])
    axes[index].set_title(class_names[label_index])
    axes[index].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# CNN architecture
model = models.Sequential(
    [
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Train
epochs = 10
batch_size = 128

history = model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=1,
)

In [ ]:
# Evaluate on held-out test images
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print("Test accuracy:", round(float(test_accuracy), 4))

In [ ]:
# Plot training curves
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Fashion-MNIST CNN")
plt.tight_layout()
plt.show()